# Automotive Supply Chain Risk GDS (Neo4j Desktop)

1. setup and connection
2. data load / graph verification
3. EDA questions as independent blocks
4. deeper analytical questions
5. GDS algorithms

## **PROBLEM STATEMENT**

### **The Challenge of Multi-Tier Automotive Supply Chains**

Modern automotive manufacturing operates through **multi-echelon supply networks** where:
- **OEMs** (Original Equipment Manufacturers) assemble final vehicles
- **Tier-1 suppliers** provide major systems (engines, transmissions, electronics)
- **Tier-2 suppliers** provide components (gears, sensors, raw materials)
- Products have **complex Bill of Materials** (a car contains ~30,000 parts)

This creates **three critical operational challenges**:

#### **1. Hidden Multi-Tier Dependencies**
When a Tier-2 supplier experiences disruption (e.g., gear manufacturer shutdown), it is difficult to quickly determine:
- Which Tier-1 systems are affected (engine, transmission)?
- Which final vehicles cannot be assembled?
- What customer orders will be delayed?
- Which alternative suppliers can provide substitute parts?

Traditional ERP systems track direct supplier relationships (OEM ↔ Tier-1) but struggle to answer: *"If this Tier-2 supplier fails, which car models are at risk?"*

#### **2. Bill of Materials Explosion**
Automotive products are **nested assemblies**:

```
Car → Engine → Cylinder Block → Pistons → Piston Rings
```

Understanding **full component traceability** requires traversing multiple levels:
- Which raw materials ultimately go into which finished cars?
- If we recall defective piston rings, which cars are affected?
- What is the total lead time from raw material to finished vehicle?

Relational databases require multiple joins; graph databases make this natural.

#### **3. Capacity Cascade Analysis**
Each supplier has **production capacity limits** and **inventory constraints**. When demand surges:
- Can Tier-2 suppliers produce enough components?
- Will Tier-1 suppliers have capacity to assemble systems?
- Where are the bottlenecks in the multi-tier network?
- How should we allocate scarce capacity across car models?

**Current Limitation:** Spreadsheets calculate single-tier capacity; multi-tier cascade requires graph traversal.

### **Why Graph Databases?**

Automotive supply chains are **inherently graphs**:
- **Nodes:** OEM, Tier-1 suppliers, Tier-2 suppliers, Products (cars, engines, gears)
- **Edges:** SUPPLIES relationships, CONTAINS (Bill of Materials), PRODUCES
- **Properties:** Lead time, capacity, inventory, demand

**Graph Analytics Enable:**
1. **Multi-hop traversal:** "Show me all Tier-2 suppliers for this car model"
2. **Impact analysis:** "If Supplier X fails, which products are affected?"
3. **Critical path analysis:** "What is the longest lead time path?"
4. **Centrality ranking:** "Which suppliers are most critical to overall production?"
5. **Community detection:** "Which suppliers form natural sourcing clusters?"

### **Research Objectives**

**Objective 1: Map Multi-Tier Supply Network Structure**
- Model OEM, Tier-1, Tier-2 suppliers as graph nodes
- Represent Bill of Materials as CONTAINS relationships
- Capture supply relationships with lead times and capacities
- Visualize end-to-end product assembly paths

**Objective 2: Identify Critical Suppliers**
- Apply PageRank to rank supplier importance across all tiers
- Compare simple degree centrality vs. weighted PageRank
- Identify single-source bottlenecks (products with one supplier)
- Prioritize supplier relationship management based on criticality

**Objective 3: Analyze Product Dependencies**
- Trace Bill of Materials from finished cars to raw components
- Identify products with complex (deep) vs. simple (shallow) BOM
- Calculate total lead time across multi-tier supply paths
- Detect products vulnerable to Tier-2 supply disruptions

**Objective 4: Discover Supply Clusters**
- Apply Louvain community detection to segment supplier network
- Characterize communities (geographic, product category, tier)
- Enable zone-based risk management (diversify across communities)
- Support scenario planning (community-level disruptions)

---

##  **DATASET STRUCTURE**

### **What the Files Contain**

**Based on Mendeley description:**

1. **Nodes (12 total):**
   - 1 OEM (car assembly plant)
   - 4 Tier-1 suppliers (engine, transmission, electronics, chassis)
   - 2 Tier-2 suppliers (components, raw materials)
   - Each node has: ID, assigned products, initial inventory, max inventory

2. **Arcs (11 total):**
   - Supply relationships between nodes
   - Properties: lead time, transport capacity per period, initial flow

3. **Products (28,049 total):**
   - Hierarchical structure: Cars → Systems → Components → Parts
   - Example: Car Model A → Engine Type X → Gear Assembly Y → Gear Z

4. **Bill of Materials:**
   - Which components are needed for each assembly
   - Quantity required per unit

5. **Customer Demand:**
   - 14 days of demand data
   - Demand per car model per day

### **Expected Graph Model**

**Nodes:**
- `OEM` (1 node)
- `Tier1Supplier` (4 nodes)
- `Tier2Supplier` (2 nodes)
- `Product` (28,049 nodes - cars, systems, components, parts)

**Relationships:**
- `SUPPLIES`: Supplier → OEM/Supplier (supply flow with lead time, capacity)
- `PRODUCES`: Supplier → Product (which supplier makes which product)
- `CONTAINS`: Product → Product (Bill of Materials - car contains engine, engine contains gears)
- `DEMANDS`: Customer → Product (demand for finished cars)

**Key Properties:**
- Nodes: inventory_current, inventory_max, tier_level, location
- SUPPLIES: lead_time_days, capacity_per_period, transport_cost
- CONTAINS: quantity_required (how many components per assembly)
- PRODUCES: production_capacity

---

### Pre-processing XLSB to CSV for Neo4j `LOAD CSV`

Because the source dataset is a single `.xlsb` workbook, we first convert it into the CSV files for data loading:
- `nodes.csv`
- `arcs.csv`
- `products.csv`
- `bom.csv`
- `node_products.csv`


In [1]:
from pathlib import Path
import sys
import json
import subprocess


def ensure_pkg(pkg: str):
    try:
        __import__(pkg)
    except ModuleNotFoundError:
        print(f"Installing missing package: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


ensure_pkg("pandas")
ensure_pkg("pyxlsb")

ROOT = Path.cwd()
if not (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from scripts.automotive_io import load_dataset, export_csv_for_neo4j

XLSB_PATH = ROOT / "files" / "2020_dataset_OfAutomotiveProductionNetwork.xlsb"
OUT_DIR = ROOT / "data" / "csv_for_neo4j"
REQUIRED = ["nodes.csv", "arcs.csv", "products.csv", "bom.csv", "node_products.csv"]

if not XLSB_PATH.is_file():
    raise FileNotFoundError(f"Workbook not found: {XLSB_PATH}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
existing = {name: (OUT_DIR / name).is_file() for name in REQUIRED}

if all(existing.values()):
    print("All required CSV files already exist. Skipping regeneration.")
else:
    missing = [k for k, v in existing.items() if not v]
    print("Missing CSV files detected:", missing)
    print("Generating CSVs from XLSB...")
    data = load_dataset(xlsb_path=XLSB_PATH)
    export_csv_for_neo4j(data, OUT_DIR)
    print("Source:", data.get("source"))
    if data.get("sheets"):
        print("Workbook sheets:", json.dumps(data["sheets"], indent=2))

print("\nCSV files status in:", OUT_DIR)
for name in REQUIRED:
    p = OUT_DIR / name
    print(f"- {name}: {'OK' if p.is_file() else 'MISSING'}")

print("Proceed with Data Loading.")

All required CSV files already exist. Skipping regeneration.

CSV files status in: c:\Users\b_nko\Downloads\Supply Chain\data\csv_for_neo4j
- nodes.csv: OK
- arcs.csv: OK
- products.csv: OK
- bom.csv: OK
- node_products.csv: OK
Proceed with Data Loading.


---
## Neo4j connection

Uses environment variables `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`.

In [2]:
import os
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from neo4j import GraphDatabase

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

ROOT = Path.cwd()
if not (ROOT / "neo4j").is_dir():
    ROOT = ROOT.parent

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query: str, params: dict | None = None) -> pd.DataFrame:
    with driver.session() as session:
        result = session.run(query, params or {})
        return pd.DataFrame([r.data() for r in result])

print("Connected:", run_query("RETURN 1 AS ok").iloc[0]["ok"])


Connected: 1


---
## **1. Data load and graph setup**

**Graph model:**
- Nodes:
    - `OEM`,
    - `Tier1Supplier`,
    - `Tier2Supplier`,
    - `Facility`,
    - `Product`,
    - `Country`
- Relationships:
    - `(:Facility)-[:SUPPLIES {lead_time_days, capacity_per_period, initial_flow, delay_probability_proxy, disruption_likelihood_proxy}]->(:Facility)`,
    - `(:Facility)-[:PRODUCES]->(:Product)`,
    - `(:Product)-[:CONTAINS {quantity_required}]->(:Product)`,
    - `(:Facility)-[:LOCATED_IN]->(:Country)`

### 1.1 Creating uniqueness constraints

This step defines **uniqueness constraints** on the IDs and names of our core node types:
- `OEM.id`
- `Tier1Supplier.id`
- `Tier2Supplier.id`
- `Facility.id`
- `Product.id`
- `Country.name`

These constraints ensure we do not accidentally create duplicate nodes for the same real-world entity when we use `MERGE` in later steps, and they also improve MATCH performance during import. We then run `SHOW CONSTRAINTS` to confirm that all constraints have been created successfully.

In [ ]:
# drop exiting nodes and relationships
run_query("MATCH (n) DETACH DELETE n")
# Create constraints and indexes 

run_query("CREATE CONSTRAINT oem_id IF NOT EXISTS FOR (o:OEM) REQUIRE o.id IS UNIQUE;")
run_query("CREATE CONSTRAINT tier1_id IF NOT EXISTS FOR (t:Tier1Supplier) REQUIRE t.id IS UNIQUE;")
run_query("CREATE CONSTRAINT tier2_id IF NOT EXISTS FOR (t:Tier2Supplier) REQUIRE t.id IS UNIQUE;")
run_query("CREATE CONSTRAINT facility_id IF NOT EXISTS FOR (f:Facility) REQUIRE f.id IS UNIQUE;")
run_query("CREATE CONSTRAINT product_id IF NOT EXISTS FOR (p:Product) REQUIRE p.id IS UNIQUE;")
run_query("CREATE CONSTRAINT country_name IF NOT EXISTS FOR (c:Country) REQUIRE c.name IS UNIQUE;")

# display constraints
df = run_query("SHOW CONSTRAINTS YIELD name, type RETURN name, type")
display(df)

In [3]:
# Connection hardening patch (run this once before loading data)
import time
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable, SessionExpired


def make_driver():
    return GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
        connection_timeout=30,
        max_connection_lifetime=1800,
        keep_alive=True,
    )


try:
    driver.close()
except Exception:
    pass

driver = make_driver()


def run_query(query: str, params: dict | None = None, timeout_s: int = 300, retries: int = 2) -> pd.DataFrame:
    global driver
    last_err = None
    for attempt in range(retries + 1):
        try:
            with driver.session() as session:
                result = session.run(query, params or {}, timeout=timeout_s)
                return pd.DataFrame([r.data() for r in result])
        except (ServiceUnavailable, SessionExpired) as e:
            last_err = e
            try:
                driver.close()
            except Exception:
                pass
            driver = make_driver()
            time.sleep(min(2 * (attempt + 1), 5))
    raise last_err

print("Connection patch active:", run_query("RETURN 1 AS ok").iloc[0]["ok"])

Connection patch active: 1


### 1.2 Loading Facility nodes

In this cell we use `LOAD CSV WITH HEADERS` to read `nodes.csv` from Neo4j's `import` directory and create facility nodes by tier (`OEM`, `Tier1Supplier`, `Tier2Supplier`, and generic `Facility`). We use `MERGE` so that re-running the cell does not create duplicates, and constraints from the previous step enforce uniqueness on IDs.

We also set core properties (`name`, `tier`, inventory fields, and coordinates) and then run a quick sample `MATCH` query to verify that facilities loaded correctly.

In [4]:
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH row WHERE toInteger(trim(row.tier)) = 0
MERGE (o:OEM:Facility {id: row.node_id})
SET o.name = row.name,
    o.inventory_current = toFloat(row.initial_inventory),
    o.inventory_max = toFloat(row.max_inventory),
    o.lat = toFloat(row.lat),
    o.lon = toFloat(row.lon)
""")

run_query("""
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH row WHERE toInteger(trim(row.tier)) = 1
MERGE (t:Tier1Supplier:Facility {id: row.node_id})
SET t.name = row.name,
    t.tier = 1,
    t.inventory_current = toFloat(row.initial_inventory),
    t.inventory_max = toFloat(row.max_inventory),
    t.lat = toFloat(row.lat),
    t.lon = toFloat(row.lon)
""")

run_query("""
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH row WHERE toInteger(trim(row.tier)) = 2
MERGE (t:Tier2Supplier:Facility {id: row.node_id})
SET t.name = row.name,
    t.tier = 2,
    t.inventory_current = toFloat(row.initial_inventory),
    t.inventory_max = toFloat(row.max_inventory),
    t.lat = toFloat(row.lat),
    t.lon = toFloat(row.lon)
""")

run_query("""
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH row WHERE toInteger(trim(row.tier)) > 2
MERGE (f:Facility {id: row.node_id})
SET f.name = row.name,
    f.tier = toInteger(trim(row.tier)),
    f.inventory_current = toFloat(row.initial_inventory),
    f.inventory_max = toFloat(row.max_inventory),
    f.lat = toFloat(row.lat),
    f.lon = toFloat(row.lon)
""")

# show 5 facilities
df = run_query("""
MATCH (f:Facility)
RETURN f.id AS id, f.name AS name, f.tier AS tier, f.inventory_current AS inventory_current
LIMIT 5
""")

display(df)

,id,name,tier,inventory_current
0,zp7,zp7,NaN,6052.0
1,zp8,zp8,NaN,0.0
2,seat-supplier_inv,seat-supplier_inv,1.0,0.0
3,seat-supplier_prod,seat-supplier_prod,1.0,6000.0
4,seat-supplier_trans,seat-supplier_trans,1.0,0.0


### 1.3 Loading Country nodes and LOCATED_IN relationships

In this cell we load unique `Country` nodes from `nodes.csv`, then connect each `Facility` to its country using `(:Facility)-[:LOCATED_IN]->(:Country)`. We use `MERGE` for both nodes and relationships so the step is safe to re-run without creating duplicates.

Finally, we return a sample of 5 facility-country links to verify the geography mapping.

In [5]:
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH DISTINCT trim(row.country) AS country
WHERE country IS NOT NULL AND country <> ''
MERGE (c:Country {name: country})
""")

run_query("""
LOAD CSV WITH HEADERS FROM 'file:///nodes.csv' AS row
WITH row WHERE trim(row.country) <> ''
MATCH (f:Facility {id: row.node_id})
MATCH (c:Country {name: trim(row.country)})
MERGE (f)-[:LOCATED_IN]->(c)
""")

# show 5 facility-country mappings
df = run_query("""
MATCH (f:Facility)-[:LOCATED_IN]->(c:Country)
RETURN f.id AS facility_id, f.name AS facility, c.name AS country
LIMIT 5
""")

display(df)

,facility_id,facility,country
0,seat-supplier_prod,seat-supplier_prod,Germany
1,engine-supplier_inv,engine-supplier_inv,Germany
2,gear-supplier_prod,gear-supplier_prod,Germany
3,battery-supplier_prod,battery-supplier_prod,Germany
4,zp8,zp8,Germany


### 1.4 Loading Product nodes

Here we import product metadata from `products.csv` and create one `Product` node per row using `MERGE`, so re-running the cell does not duplicate products. We set `name`, `type`, and `level` (with a safe fallback to `-1` when level is blank).

We then return 5 sample products to confirm that the product attributes loaded correctly.

In [ ]:
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///products.csv' AS row
MERGE (p:Product {id: row.product_id})
SET p.name = row.name,
    p.type = row.product_type,
    p.level = CASE
      WHEN row.level IS NULL OR trim(toString(row.level)) = '' THEN -1
      ELSE toInteger(trim(toString(row.level)))
    END
""")

# show 5 products
df = run_query("""
MATCH (p:Product)
RETURN p.id AS product_id, p.name AS name, p.type AS type, p.level AS level
LIMIT 5
""")

display(df)

### 1.5 Loading SUPPLIES relationships

This step imports `arcs.csv` and creates `(:Facility)-[:SUPPLIES]->(:Facility)` relationships. We `MATCH` existing facilities and `MERGE` each edge so re-running the import remains idempotent. We also set operational and risk proxy properties (`lead_time_days`, `capacity_per_period`, `initial_flow`, `delay_probability_proxy`, and `disruption_likelihood_proxy`).

Finally, we return 5 sample SUPPLIES edges to validate relationship wiring and properties.

In [ ]:
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///arcs.csv' AS row
MATCH (a:Facility {id: row.from_node})
MATCH (b:Facility {id: row.to_node})
MERGE (a)-[r:SUPPLIES]->(b)
SET r.lead_time_days = toFloat(row.lead_time_days),
    r.capacity_per_period = toFloat(row.capacity_per_period),
    r.initial_flow = toFloat(row.initial_flow),
    r.delay_probability_proxy = toFloat(row.delay_probability_proxy),
    r.disruption_likelihood_proxy = toFloat(row.disruption_likelihood_proxy)
""")

# show 5 SUPPLIES relationships
df = run_query("""
MATCH (a:Facility)-[r:SUPPLIES]->(b:Facility)
RETURN a.id AS from_node, b.id AS to_node, r.lead_time_days AS lead_time_days, r.capacity_per_period AS capacity
LIMIT 5
""")

display(df)

### 1.6 Loading BOM (CONTAINS relationships)

In this cell we import `bom.csv` and build `(:Product)-[:CONTAINS]->(:Product)` edges representing the Bill of Materials hierarchy. We use `MATCH` on previously loaded products and `MERGE` to avoid duplicate BOM edges on re-run.

We then return 5 sample BOM links and quantities to verify structural correctness.

In [ ]:
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///bom.csv' AS row
MATCH (parent:Product {id: row.parent_id})
MATCH (child:Product {id: row.child_id})
MERGE (parent)-[r:CONTAINS]->(child)
SET r.quantity_required = toFloat(row.quantity)
""")

# show 5 BOM relationships
df = run_query("""
MATCH (parent:Product)-[r:CONTAINS]->(child:Product)
RETURN parent.id AS parent_id, child.id AS child_id, r.quantity_required AS quantity_required
LIMIT 5
""")

display(df)

### 1.7 Loading PRODUCES relationships

This step imports `node_products.csv` and creates `(:Facility)-[:PRODUCES]->(:Product)` links. Using `MERGE` ensures one relationship per facility-product pair if the cell is executed multiple times.

Finally, we sample 5 produced products to validate that production assignments are loaded correctly.

In [ ]:
run_query("""
LOAD CSV WITH HEADERS FROM 'file:///node_products.csv' AS row
MATCH (f:Facility {id: row.node_id})
MATCH (p:Product {id: row.product_id})
MERGE (f)-[:PRODUCES]->(p)
""")

# show 5 PRODUCES relationships
df = run_query("""
MATCH (f:Facility)-[:PRODUCES]->(p:Product)
RETURN f.id AS facility_id, f.name AS facility, p.id AS product_id, p.type AS product_type
LIMIT 5
""")

display(df)

### 1.8 Verification

This verification step confirms that major labels and relationship types have been loaded as expected. It provides a quick structural sanity check before moving into EDA and GDS analysis.

In [ ]:
df_labels = run_query("""
MATCH (n)
UNWIND labels(n) AS label
RETURN label, count(*) AS count
ORDER BY count DESC
""")

df_rels = run_query("""
MATCH ()-[r]->()
RETURN type(r) AS rel_type, count(*) AS count
ORDER BY count DESC
""")

display(df_labels)
display(df_rels)

---


## 2. EDA Questions


### 2.1 Total nodes and relationships by label/type


In [ ]:
df = run_query("""\nMATCH (n)
UNWIND labels(n) AS label
RETURN label, count(*) AS count
ORDER BY count DESC\n""")
display(df.head(20))


**Explanation**

Counts graph objects by label to confirm scope and completeness of ingestion.


**Interpretation / Commentary**

Use this as a baseline quality check. The `Product` label should dominate volume.


### 2.2 Product hierarchy distribution by type


In [ ]:
df = run_query("""\nMATCH (p:Product)
RETURN p.type AS product_type, count(*) AS count
ORDER BY count DESC\n""")
display(df.head(20))


**Explanation**

Profiles product mix (car/system/component/part categories).


**Interpretation / Commentary**

A heavy tail of components/parts indicates deep assembly dependency.


### 2.3 Product hierarchy distribution by level


In [ ]:
df = run_query("""\nMATCH (p:Product)
RETURN p.level AS level, count(*) AS count
ORDER BY level\n""")
display(df.head(20))


**Explanation**

Checks depth of BOM hierarchy encoded in product levels.


**Interpretation / Commentary**

Higher concentrations at deeper levels imply greater upstream risk propagation.


### 2.4 Top facilities by number of products produced


In [ ]:
df = run_query("""\nMATCH (f:Facility)-[:PRODUCES]->(p:Product)
RETURN f.name AS facility, f.tier AS tier, count(p) AS product_count
ORDER BY product_count DESC\n""")
display(df.head(20))


**Explanation**

Measures production portfolio concentration across facilities.


**Interpretation / Commentary**

High-count facilities are operationally important and candidate risk hotspots.


### 2.5 Single-sourced products (bottlenecks)


In [ ]:
df = run_query("""\nMATCH (p:Product)<-[:PRODUCES]-(f:Facility)
WITH p, count(DISTINCT f) AS supplier_count
WHERE supplier_count = 1
RETURN p.id AS product_id, p.type AS product_type, supplier_count
LIMIT 50\n""")
display(df.head(20))


**Explanation**

Detects products with only one producing facility.


**Interpretation / Commentary**

Single-source products are direct points of failure under disruption.


### 2.6 Lead-time statistics by supplier tier


In [ ]:
df = run_query("""\nMATCH (f:Facility)-[s:SUPPLIES]->(:Facility)
RETURN f.tier AS tier,
       avg(s.lead_time_days) AS avg_lead_time,
       min(s.lead_time_days) AS min_lead_time,
       max(s.lead_time_days) AS max_lead_time
ORDER BY tier\n""")
display(df.head(20))


**Explanation**

Summarizes transport/production delay behavior by echelon.


**Interpretation / Commentary**

Longer tier lead times increase cycle-time risk and planning buffer requirements.


### 2.7 Capacity distribution by supplier tier


In [ ]:
df = run_query("""\nMATCH (f:Facility)-[s:SUPPLIES]->(:Facility)
RETURN f.tier AS tier, sum(s.capacity_per_period) AS total_capacity
ORDER BY tier\n""")
display(df.head(20))


**Explanation**

Aggregates available throughput to reveal where network capacity sits.


**Interpretation / Commentary**

Low capacity with high dependency signals likely bottlenecks.


### 2.8 Inventory utilization at facilities


In [ ]:
df = run_query("""\nMATCH (f:Facility)
WHERE f.inventory_max IS NOT NULL AND f.inventory_max > 0
RETURN f.name AS facility,
       f.tier AS tier,
       f.inventory_current AS current_inv,
       f.inventory_max AS max_inv,
       toFloat(f.inventory_current) / toFloat(f.inventory_max) AS utilization
ORDER BY utilization DESC\n""")
display(df.head(20))


**Explanation**

Compares current inventory to available capacity.


**Interpretation / Commentary**

Near-1.0 utilization suggests reduced resilience to demand or supply shocks.


### 2.9 Products produced by tier and type


In [ ]:
df = run_query("""\nMATCH (f:Facility)-[:PRODUCES]->(p:Product)
RETURN f.tier AS tier, p.type AS product_type, count(*) AS n
ORDER BY tier, n DESC\n""")
display(df.head(20))


**Explanation**

Maps specialization across echelons.


**Interpretation / Commentary**

Clear specialization supports modular sourcing strategy but can increase coupling.


### 2.10 Highest-risk supply arcs by proxy scores


In [ ]:
df = run_query("""\nMATCH (a:Facility)-[s:SUPPLIES]->(b:Facility)
RETURN a.name AS from_facility,
       b.name AS to_facility,
       s.delay_probability_proxy AS delay_proxy,
       s.disruption_likelihood_proxy AS disruption_proxy,
       s.delay_probability_proxy * s.disruption_likelihood_proxy AS risk_score
ORDER BY risk_score DESC
LIMIT 30\n""")
display(df.head(20))


**Explanation**

Ranks arcs by a composite risk proxy to prioritize monitoring.


**Interpretation / Commentary**

Top arcs should be first candidates for contingency planning and dual-routing.


---


## 3. Deeper Analytical Questions


### 3.1 Critical path style exposure (multi-hop supply)


In [ ]:
df = run_query("""
MATCH path = (t2:Tier2Supplier)-[:SUPPLIES*1..4]->(o:OEM)
WITH path,
     reduce(total = 0.0, r IN relationships(path) | total + coalesce(r.lead_time_days, 0.0)) AS total_lead_time
RETURN [n IN nodes(path) | coalesce(n.name, n.id)] AS path_nodes, total_lead_time
ORDER BY total_lead_time DESC
LIMIT 25
""")
display(df)


**Explanation**

Evaluates cumulative lead time across multi-tier routes.


**Interpretation / Commentary**

Long paths with high cumulative lead time are candidates for safety stock, route redesign, or supplier diversification.


### 3.2 Cars with high single-point-of-failure exposure


In [ ]:
df = run_query("""
MATCH (car:Product {type:'car'})-[:CONTAINS*1..6]->(comp:Product)
MATCH (comp)<-[:PRODUCES]-(f:Facility)
WITH car, comp, count(DISTINCT f) AS supplier_count
WHERE supplier_count = 1
RETURN car.id AS car_id, count(DISTINCT comp) AS vulnerable_components
ORDER BY vulnerable_components DESC
LIMIT 25
""")
display(df)


**Explanation**

Counts downstream components in each car that are single-sourced.


**Interpretation / Commentary**

Cars with larger vulnerable component counts have higher stoppage risk under localized disruptions.


---


## 4. GDS Workflow (Neo4j Desktop)


### 4.1 PageRank: critical facilities/products


In [ ]:
df = run_query("""
CALL gds.pageRank.stream('automotive-network')
YIELD nodeId, score
RETURN coalesce(gds.util.asNode(nodeId).name, gds.util.asNode(nodeId).id) AS node,
       labels(gds.util.asNode(nodeId)) AS labels,
       score
ORDER BY score DESC
LIMIT 20
""")
display(df)


**Explanation**

PageRank scores nodes by importance based on recursive dependency.


**Interpretation / Commentary**

Top-ranked suppliers and products are strategic critical nodes; they should receive stronger risk controls.


### 4.2 Louvain: dependency communities


In [ ]:
df = run_query("""
CALL gds.louvain.stream('automotive-network')
YIELD nodeId, communityId
RETURN communityId,
       labels(gds.util.asNode(nodeId))[0] AS node_type,
       count(*) AS members
ORDER BY members DESC
""")
display(df.head(30))


**Explanation**

Louvain partitions nodes into tightly connected communities.


**Interpretation / Commentary**

Community-level concentration can guide diversification strategy across independent clusters.


---


## 5. Closing


This notebook now matches the recommender-project style:

- independent question cells
- explicit explanation cells
- explicit interpretation/commentary cells
- Neo4j Desktop first workflow for Cypher and GDS.
